### Análise Exploratória das Métricas Globais de Conectividade

Este notebook apresenta a análise exploratória das métricas globais de conectividade funcional cerebral extraídas a partir de sinais de EEG em estado de repouso.

O objetivo desta etapa é:
- caracterizar estatisticamente as métricas globais,
- investigar diferenças entre condições experimentais (EO vs EC),
- fornecer subsídios para as etapas posteriores de redução de dimensionalidade e aprendizado de máquina.


### Import e carregamento

In [41]:
import pandas as pd
import numpy as np
import plotly.io as pio

import plotly.express as px
import plotly.graph_objects as go


In [42]:
pio.templates.default = "plotly_dark"

In [20]:
# Carrega dataset final
df_dataset = pd.read_parquet("../data/processed/dataset_features.parquet")

df_dataset.shape


(165, 393)

In [21]:
df_dataset['condition'].value_counts()

condition
EO    82
EC    82
E0     1
Name: count, dtype: int64

In [22]:
df_dataset[df_dataset['condition'] == 'E0']

,degree_0,degree_1,degree_2,degree_3,degree_4,degree_5,degree_6,degree_7,degree_8,degree_9,...,SKID_Diagnoses 1,SKID_Diagnoses 2,Comments_SKID_assessment,Hamilton_Scale,BSL23_sumscore,BSL23_behavior,AUDIT,Standard_Alcoholunits_Last_28days,Alcohol_Dependence_In_1st-3rd_Degree_relative,Relationship_Status
157,43445.451921,41030.964062,44195.097088,43339.644809,38800.674257,42310.860871,42684.610963,46124.215349,43205.4517,39891.504851,...,#CODE:V71.09 #DESC:No Diagnosis or Condition ...,None,None,0.0,1.0,0.0,3.0,"13,5",No,Yes


Removendo o E0 do banco:

In [23]:
df_dataset = df_dataset[~(df_dataset["condition"] == "E0")]

In [24]:
df_dataset[df_dataset['condition'] == 'E0']

,degree_0,degree_1,degree_2,degree_3,degree_4,degree_5,degree_6,degree_7,degree_8,degree_9,...,SKID_Diagnoses 1,SKID_Diagnoses 2,Comments_SKID_assessment,Hamilton_Scale,BSL23_sumscore,BSL23_behavior,AUDIT,Standard_Alcoholunits_Last_28days,Alcohol_Dependence_In_1st-3rd_Degree_relative,Relationship_Status


### Seleção das métricas globais

“As métricas globais representam propriedades agregadas da rede funcional cerebral, independententes de regiões específicas, refletindo o nível geral de integração e variabilidade da conectividade.”

In [30]:
GLOBAL_FEATURES = [
    "global_efficiency",
    "edges_mean",
    "edges_std",
    "edges_cv"
]

id_cols = ["subject_id", "condition"]

df_global = df_dataset[id_cols + GLOBAL_FEATURES]

df_global.head()

,subject_id,condition,global_efficiency,edges_mean,edges_std,edges_cv
0,sub-010060,EO,1.0,1830.0,0.0,0.0
1,sub-010061,EC,1.0,1770.0,0.0,0.0
2,sub-010061,EO,1.0,1770.0,0.0,0.0
3,sub-010062,EC,1.0,1653.0,0.0,0.0
4,sub-010062,EO,1.0,1770.0,0.0,0.0


In [32]:
df_global[GLOBAL_FEATURES].describe()

,global_efficiency,edges_mean,edges_std,edges_cv
count,164.0,164.000000,164.0,164.0
mean,1.0,1768.530488,0.0,0.0
std,0.0,70.068628,0.0,0.0
min,1.0,1540.000000,0.0,0.0
25%,1.0,1711.000000,0.0,0.0
50%,1.0,1770.000000,0.0,0.0
75%,1.0,1830.000000,0.0,0.0
max,1.0,1830.000000,0.0,0.0


In [33]:
df_global.groupby("condition")[GLOBAL_FEATURES].describe()


global_efficiency                                    edges_mean  \
                      count mean  std  min  25%  50%  75%  max      count   
condition                                                                   
EC                     82.0  1.0  0.0  1.0  1.0  1.0  1.0  1.0       82.0   
EO                     82.0  1.0  0.0  1.0  1.0  1.0  1.0  1.0       82.0   

                        ... edges_std      edges_cv                           \
                  mean  ...       75%  max    count mean  std  min  25%  50%   
condition               ...                                                    
EC         1766.451220  ...       0.0  0.0     82.0  0.0  0.0  0.0  0.0  0.0   
EO         1770.609756  ...       0.0  0.0     82.0  0.0  0.0  0.0  0.0  0.0   

                     
           75%  max  
condition            
EC         0.0  0.0  
EO         0.0  0.0  

[2 rows x 32 columns]

### Boxplots EO vs EC

Eficiência global

In [43]:
fig = px.box(
    df_global,
    x="condition",
    y="global_efficiency",
    color="condition",
    points="all",
    title="Eficiência Global da Rede por Condição Experimental"
)
fig.show()

Número médio de arestas

In [44]:
fig = px.box(
    df_global,
    x="condition",
    y="edges_mean",
    color="condition",
    points="all",
    title="Número Médio de Arestas por Condição"
)

fig.show()


### Variabilidade da conectividade

In [45]:
fig = px.box(
    df_global,
    x="condition",
    y="edges_cv",
    color="condition",
    points="all",
    title="Coeficiente de Variação da Conectividade ao Longo do Tempo"
)
fig.show()


### Correlação entre métricas globais

In [39]:
corr = df_global[GLOBAL_FEATURES].corr()
corr

,global_efficiency,edges_mean,edges_std,edges_cv
global_efficiency,NaN,NaN,NaN,NaN
edges_mean,NaN,1.0,NaN,NaN
edges_std,NaN,NaN,NaN,NaN
edges_cv,NaN,NaN,NaN,NaN


In [46]:
fig = px.imshow(
    corr,
    text_auto=True,
    title="Matriz de Correlação entre Métricas Globais",
    color_continuous_scale="RdBu_r",
    aspect="auto"
)
fig.show()
